In [342]:
# 5) Dataset hazırlığı (TS/C2C -> orta dilim 2D + DataLoader)
import pandas as pd
from monai.transforms import (LoadImageD, EnsureChannelFirstD, ScaleIntensityRanged, RandFlipD, RandRotateD, RandZoomD, EnsureTypeD, Lambdad, ResizeD, Compose)
from monai.data import CacheDataset, DataLoader
import torch

rows=[]
MERGED_DIR = DATA_ROOT/'labels_merged'
if MERGED_DIR.exists():
    for pL in MERGED_DIR.glob('*_psoas_left.nii.gz'):
        cid=pL.name.replace('_psoas_left.nii.gz','')
        pR=MERGED_DIR/f'{cid}_psoas_right.nii.gz'
        img = IMAGES_DIR/f'{cid}.nii.gz'
        if not img.exists(): img = IMAGES_DIR/f'{cid}.nii'
        if pR.exists() and img.exists():
            rows.append({'case_id':cid,'image':str(img),'psoas_left':str(pL),'psoas_right':str(pR)})
if rows:
    df=pd.DataFrame(rows); DATASET_CSV=DATA_ROOT/'dataset_merged.csv'; df.to_csv(DATASET_CSV, index=False)
    print('[DATASET] Using MERGED labels. Entries:', len(df))
else:
    df=pd.DataFrame(); print('[DATASET] No merged labels found; (simplified pipeline)')

_take_mid = lambda x: x[..., x.shape[-1]//2] if x.ndim>=3 else x
if df.empty:
    train_ds=val_ds=[]; train_loader=val_loader=None
    print('[DATASET] Empty – add NIfTI files.')
else:
    items=[{'image':r['image'],'psoas_left':r['psoas_left'],'psoas_right':r['psoas_right']} for _,r in df.iterrows()]
    val_count=max(1,int(0.2*len(items)))
    train_files=items[:-val_count] if val_count < len(items) else items
    val_files=items[-val_count:] if val_count < len(items) else items
    train_transforms=Compose([
        LoadImageD(keys=['image','psoas_left','psoas_right']),
        EnsureChannelFirstD(keys=['image','psoas_left','psoas_right']),
        Lambdad(keys=['image','psoas_left','psoas_right'], func=_take_mid),
        ScaleIntensityRanged(keys=['image'], a_min=-1000,a_max=1000,b_min=0.0,b_max=1.0, clip=True),
        ResizeD(keys=['image'], spatial_size=(256,256), mode='bilinear'),
        ResizeD(keys=['psoas_left','psoas_right'], spatial_size=(256,256), mode='nearest'),
        RandFlipD(keys=['image','psoas_left','psoas_right'], prob=0.5, spatial_axis=0),
        RandRotateD(keys=['image','psoas_left','psoas_right'], range_x=0.1, prob=0.3),
        RandZoomD(keys=['image','psoas_left','psoas_right'], min_zoom=0.9,max_zoom=1.1,prob=0.3),
        EnsureTypeD(keys=['image','psoas_left','psoas_right'])
    ])
    val_transforms=Compose([
        LoadImageD(keys=['image','psoas_left','psoas_right']),
        EnsureChannelFirstD(keys=['image','psoas_left','psoas_right']),
        Lambdad(keys=['image','psoas_left','psoas_right'], func=_take_mid),
        ScaleIntensityRanged(keys=['image'], a_min=-1000,a_max=1000,b_min=0.0,b_max=1.0, clip=True),
        ResizeD(keys=['image'], spatial_size=(256,256), mode='bilinear'),
        ResizeD(keys=['psoas_left','psoas_right'], spatial_size=(256,256), mode='nearest'),
        EnsureTypeD(keys=['image','psoas_left','psoas_right'])
    ])
    batch_size=int(CONFIG.get('BATCH_SIZE', 4 if torch.cuda.is_available() else 2)) if CONFIG.get('BATCH_SIZE') is not None else (4 if torch.cuda.is_available() else 2)
    num_workers=int(CONFIG.get('NUM_WORKERS',2))
    pin_mem=torch.cuda.is_available(); pers= num_workers>0
    train_ds=CacheDataset(train_files, transform=train_transforms, cache_rate=0.0)
    val_ds=CacheDataset(val_files, transform=val_transforms, cache_rate=0.0)
    train_loader=DataLoader(train_ds,batch_size=batch_size,shuffle=True,num_workers=num_workers,pin_memory=pin_mem,persistent_workers=pers)
    val_loader=DataLoader(val_ds,batch_size=batch_size,shuffle=False,num_workers=num_workers,pin_memory=pin_mem,persistent_workers=pers)
    print('[DATASET] Sizes:', len(train_ds), len(val_ds), '| batch_size=', batch_size)


[DATASET] No merged labels found; (simplified pipeline)
[DATASET] Empty – add NIfTI files.


In [343]:
# 4) TotalSegmentator teacher üretimi + label merge
import sys, subprocess, json, nibabel as nib, numpy as np
from pathlib import Path
from tqdm import tqdm
TS_TASKS=['abdominal_muscles','total']
image_paths = sorted([p for p in IMAGES_DIR.glob('*.nii*')])
max_cases = CONFIG.get('MAX_CASES', None)
if max_cases is not None:
    image_paths = image_paths[:int(max_cases)]
print('[TS] Processing', len(image_paths), 'cases')

def run_ts(inp: Path, out_dir: Path, task: str):
    out_dir.mkdir(parents=True, exist_ok=True)
    marker = out_dir/f'{task}_DONE'
    if marker.exists():
        return
    cmd=[sys.executable,'-m','totalsegmentator','-i',str(inp),'-o',str(out_dir),'-ta',task,'--fast']
    if task=='total':
        cmd += ['--roi_subset','torso']
    print('[RUN]', ' '.join(cmd))
    subprocess.check_call(cmd)
    marker.touch()

for img in tqdm(image_paths, desc='TS tasks'):
    cid = img.name.replace('.nii.gz','').replace('.nii','')
    outd = TS_OUT/cid
    for t in TS_TASKS:
        run_ts(img, outd, t)
print('[TS] Done')

MERGED_DIR = DATA_ROOT/'labels_merged'
MERGED_DIR.mkdir(parents=True, exist_ok=True)
MERGE_STRATEGY = CONFIG.get('MERGE_STRATEGY','TS_ONLY')
C2C_CANDIDATES=[('psoas_major_left.nii.gz','psoas_major_right.nii.gz'),('psoas_left.nii.gz','psoas_right.nii.gz'),('psoas_l.nii.gz','psoas_r.nii.gz')]

from pathlib import Path as _Path
import numpy as _np
import nibabel as _nib

def _load_bin(path:_Path):
    arr=_nib.load(str(path)).get_fdata(); return (arr>0).astype(_np.uint8)

def _save_like(ref_path:_Path, mask:_np.ndarray, out_path:_Path):
    ref=_nib.load(str(ref_path)); _nib.save(_nib.Nifti1Image(mask.astype(_np.uint8), ref.affine, ref.header), str(out_path))

cases = [p.name.replace('.nii.gz','').replace('.nii','') for p in image_paths]
print('[MERGE] Strategy=', MERGE_STRATEGY)
for cid in cases:
    ts_dir = TS_OUT/cid
    left_ts = ts_dir/'psoas_major_left.nii.gz'
    right_ts= ts_dir/'psoas_major_right.nii.gz'
    # C2C opsiyonel
    c2c_case = C2C_DIR/cid
    left_c2c = right_c2c = None
    if c2c_case.exists():
        for a,b in C2C_CANDIDATES:
            if (c2c_case/a).exists() and (c2c_case/b).exists():
                left_c2c, right_c2c = c2c_case/a, c2c_case/b
                break
    def choose():
        if MERGE_STRATEGY=='TS_ONLY': return left_ts, right_ts
        if MERGE_STRATEGY=='C2C_ONLY' and left_c2c and right_c2c: return left_c2c, right_c2c
        if MERGE_STRATEGY in ('UNION','INTERSECT') and left_c2c and right_c2c and left_ts.exists() and right_ts.exists():
            tsL=_load_bin(left_ts); tsR=_load_bin(right_ts); cL=_load_bin(left_c2c); cR=_load_bin(right_c2c)
            if MERGE_STRATEGY=='UNION':
                mL=((tsL+cL)>0).astype(_np.uint8); mR=((tsR+cR)>0).astype(_np.uint8)
            else:
                mL=((tsL*cL)>0).astype(_np.uint8); mR=((tsR*cR)>0).astype(_np.uint8)
            _save_like(left_ts, mL, MERGED_DIR/f'{cid}_psoas_left.nii.gz')
            _save_like(right_ts, mR, MERGED_DIR/f'{cid}_psoas_right.nii.gz')
            return None, None
        return left_ts, right_ts
    useL,useR = choose()
    if useL:
        _save_like(useL, _load_bin(useL), MERGED_DIR/f'{cid}_psoas_left.nii.gz')
    if useR:
        _save_like(useR, _load_bin(useR), MERGED_DIR/f'{cid}_psoas_right.nii.gz')
print('[MERGE] Done →', MERGED_DIR)


[TS] Processing 0 cases


TS tasks: 0it [00:00, ?it/s]

TS tasks: 0it [00:00, ?it/s]

[TS] Done
[MERGE] Strategy= TS_ONLY
[MERGE] Done → data/labels_merged


In [344]:
# 3) GPU & Drive mount + NIfTI toplama
import torch, shutil, os
from pathlib import Path
from datetime import datetime
DATA_ROOT = Path('/content/data') if IN_COLAB else Path('data')
IMAGES_DIR = DATA_ROOT/'images'
TS_OUT = DATA_ROOT/'ts_labels'
C2C_DIR = DATA_ROOT/'c2c_labels'
OUT_RUNS = Path('/content/outputs') if IN_COLAB else Path('outputs')
for p in [DATA_ROOT, IMAGES_DIR, TS_OUT, C2C_DIR, OUT_RUNS]:
    p.mkdir(parents=True, exist_ok=True)
print('[GPU]', 'CUDA available' if torch.cuda.is_available() else 'CPU mode')
if IN_COLAB and CONFIG.get('USE_DRIVE', True):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('[DRIVE] Mounted')
        drive_root = Path(CONFIG['DRIVE_DATA_ROOT'])
        method = CONFIG.get('COPY_FROM_DRIVE','symlink')
        subdirs = CONFIG.get('DRIVE_IMAGES_SUBDIRS',['images','imagesTr','imagesTs'])
        added=0
        if drive_root.exists():
            for sd in subdirs:
                cand = drive_root/sd
                if cand.exists():
                    for f in cand.rglob('*'):
                        if f.is_file() and (f.suffix=='.nii' or f.name.endswith('.nii.gz')):
                            tgt = IMAGES_DIR/f.name
                            if not tgt.exists():
                                try:
                                    if method=='symlink':
                                        os.symlink(f, tgt)
                                    else:
                                        shutil.copy2(f, tgt)
                                    added+=1
                                except Exception as e:
                                    print('[DATA] Skip', f, '→', e)
            print(f'[DATA] Added {added} image links/files from Drive')
        else:
            print('[WARN] DRIVE_DATA_ROOT not found:', drive_root)
    except Exception as e:
        print('[DRIVE] Mount skipped:', e)
print('[INFO] Images now:', len(list(IMAGES_DIR.glob('*.nii*'))))


[GPU] CPU mode
[INFO] Images now: 0


In [345]:
# 2) Ortam kurulumu (Colab tespiti + paketler)
import sys, subprocess, os, platform
IN_COLAB = 'google.colab' in sys.modules
reqs = ['torch','torchvision','torchaudio','monai','tqdm','pandas','matplotlib','seaborn','nibabel','pydicom','opencv-python','scikit-image','SimpleITK','totalsegmentator','ipywidgets']
if IN_COLAB:
    print('[SETUP] Installing base packages...')
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--upgrade','pip'])
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet'] + reqs)
else:
    print('[INFO] Not in Colab; ensure dependencies installed locally.')
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())


[INFO] Not in Colab; ensure dependencies installed locally.
Python: 3.11.8 | Platform: macOS-15.6-arm64-arm-64bit


In [346]:
# 1) CONFIG – hızlı test varsayılanları
CONFIG = {
    'USE_DRIVE': True,
    'DRIVE_DATA_ROOT': '/content/drive/MyDrive/AMOS22',
    'DRIVE_IMAGES_SUBDIRS': ['images', 'imagesTr', 'imagesTs'],
    'COPY_FROM_DRIVE': 'symlink',  # veya 'copy'
    'MAX_CASES': 3,        # Hızlı test: 3 | Tam: None
    'MAX_EPOCHS': 5,       # Hızlı test: 5 | Tam: 60
    'MERGE_STRATEGY': 'TS_ONLY',  # 'TS_ONLY'|'C2C_ONLY'|'UNION'|'INTERSECT'
    'BATCH_SIZE': None,    # otomatik seçim; override edilebilir
    'NUM_WORKERS': 2,
    'USE_AMP': True        # Mixed precision
}
print('CONFIG loaded:', CONFIG)


CONFIG loaded: {'USE_DRIVE': True, 'DRIVE_DATA_ROOT': '/content/drive/MyDrive/AMOS22', 'DRIVE_IMAGES_SUBDIRS': ['images', 'imagesTr', 'imagesTs'], 'COPY_FROM_DRIVE': 'symlink', 'MAX_CASES': 3, 'MAX_EPOCHS': 5, 'MERGE_STRATEGY': 'TS_ONLY', 'BATCH_SIZE': None, 'NUM_WORKERS': 2, 'USE_AMP': True}


# Colab Pro+ L3 Psoas (VFA/PMA Pipeline) – Yeniden Yükleme

Bu notebook önceki içerikten boş görünüyorsa hücreler yeniden oluşturulmuştur.
Aşağıdaki akış:
1. CONFIG tanımı
2. Ortam & paket kurulumu
3. GPU ve Drive mount
4. TotalSegmentator teacher üretimi
5. Dataset hazırlığı (3D→2D orta slice, 256×256)
6. UNet + AMP eğitim loop (hızlı test varsayılanları)
7. Overlay görselleştirme
8. Model export + TorchScript

Hızlı test: 3 vaka / 5 epoch (≈10 dk). Tam eğitim: tüm vakalar / 60 epoch.


In [347]:
# 6) Eğitim konfigürasyonu (UNet + DiceCE + AMP)
import torch, torch.nn as nn, torch.optim as optim
from monai.networks.nets import UNet
from monai.losses import DiceCELoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=2,           # left/right psoas için iki sınıf (kanal)
    channels=(16,32,64),      # hafif model (hızlı test). Tam eğitimde (32,64,128,256) genişletilebilir.
    strides=(2,2),
    num_res_units=2,
).to(device)
max_epochs = int(CONFIG.get('MAX_EPOCHS', 60))
lr = 1e-3
use_amp = bool(CONFIG.get('USE_AMP', torch.cuda.is_available()))
optimizer = optim.AdamW(model.parameters(), lr=lr)
criterion = DiceCELoss(sigmoid=False, to_onehot_y=True, softmax=True)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
print('[TRAIN] max_epochs=', max_epochs, '| AMP=', use_amp, '| device=', device)


[TRAIN] max_epochs= 5 | AMP= True | device= cpu


/var/folders/pb/dhybj9093tg6rwshg64043lh0000gn/T/ipykernel_1668/515785422.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/Users/alperenogras/Desktop/L3_SO_ANALYSIS/.venv/lib/python3.11/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [348]:
# 7) Eğitim döngüsü
from tqdm.auto import tqdm
import numpy as np, math
ckpt_dir = OUT_RUNS/'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
best_val = math.inf
if train_loader is None or val_loader is None or len(train_ds)==0:
    print('[TRAIN] Dataset boş; eğitim atlandı. Veri ekleyip dataset hücresini yeniden çalıştırın.')
else:
    for epoch in range(1, max_epochs+1):
        model.train(); train_losses=[]
        for batch in train_loader:
            x = batch['image'].to(device)
            y_l = batch['psoas_left'].to(device)
            y_r = batch['psoas_right'].to(device)
            y = torch.cat([y_l, y_r], dim=1)  # [B,2,H,W]
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            train_losses.append(loss.item())
        model.eval(); val_losses=[]
        with torch.no_grad():
            for batch in val_loader:
                x = batch['image'].to(device)
                y_l = batch['psoas_left'].to(device)
                y_r = batch['psoas_right'].to(device)
                y = torch.cat([y_l, y_r], dim=1)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    logits = model(x); loss = criterion(logits, y)
                val_losses.append(loss.item())
        tl = np.mean(train_losses) if train_losses else float('nan')
        vl = np.mean(val_losses) if val_losses else float('nan')
        print(f'Epoch {epoch:03d} | train={tl:.4f} val={vl:.4f}')
        if vl < best_val:
            best_val = vl
            torch.save({'epoch':epoch,'model':model.state_dict(),'val_loss':vl}, ckpt_dir/'best.pt')
            print('[CKPT] best.pt güncellendi')
    torch.save({'epoch':max_epochs,'model':model.state_dict(),'val_loss':vl}, ckpt_dir/'last.pt')
    print('[TRAIN] Tamamlandı')


[TRAIN] Dataset boş; eğitim atlandı. Veri ekleyip dataset hücresini yeniden çalıştırın.


In [349]:
# 8) Overlay görselleştirme (örnek)
import matplotlib.pyplot as plt
import numpy as np

def show_overlay(x_tensor, pred_logits, idx=0):
    base = x_tensor[idx,0].detach().cpu().numpy()
    sm = torch.softmax(pred_logits[idx], dim=0).detach().cpu().numpy()  # [2,H,W]
    left = sm[0] > 0.5; right = sm[1] > 0.5
    norm = (base - base.min())/(base.ptp()+1e-6)
    rgb = np.stack([norm, norm, norm], axis=-1)
    rgb[...,2] = np.maximum(rgb[...,2], left.astype(float)*0.9)   # Blue
    rgb[...,1] = np.maximum(rgb[...,1], right.astype(float)*0.9)  # Green
    plt.figure(figsize=(5,5))
    plt.imshow(rgb); plt.title('Overlay (Blue=Left, Green=Right)')
    plt.axis('off'); plt.show()

try:
    if val_loader and len(val_ds)>0:
        batch = next(iter(val_loader))
        x = batch['image'].to(device)
        model.eval();
        with torch.no_grad():
            pred = model(x)
        show_overlay(x, pred, idx=0)
    else:
        print('[OVERLAY] Veri yok, atlandı.')
except Exception as e:
    print('[OVERLAY] Hata:', e)


[OVERLAY] Veri yok, atlandı.


In [350]:
# 9) TotalSegmentator cache & C2C entegrasyon notu
from pathlib import Path
TS_CACHE = Path.home()/'.totalsegmentator'
print('[TS CACHE]', 'exists' if TS_CACHE.exists() else 'missing (indirilecek)')
print('[C2C] MERGE_STRATEGY=', CONFIG.get('MERGE_STRATEGY'))
print('C2C maskeleri varsa C2C_DIR içinde case klasörü + psoas_left/right adlandırmaları ile yerleştirin.')


[TS CACHE] exists
[C2C] MERGE_STRATEGY= TS_ONLY
C2C maskeleri varsa C2C_DIR içinde case klasörü + psoas_left/right adlandırmaları ile yerleştirin.


In [351]:
# 10) Model export + TorchScript
from datetime import datetime
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
export_dir = OUT_RUNS/f'model_export_{stamp}'
export_dir.mkdir(parents=True, exist_ok=True)
for ck in ['best.pt','last.pt']:
    src = ckpt_dir/ck
    if src.exists():
        shutil.copy2(src, export_dir/ck)
print('[EXPORT] Saved checkpoints to', export_dir)
# TorchScript
scripted_path = ckpt_dir/'model_scripted.pt'
try:
    model.eval()
    example = torch.randn(1,1,256,256, device=device)
    traced = torch.jit.trace(model, example)
    torch.jit.save(traced, str(scripted_path))
    shutil.copy2(scripted_path, export_dir/'model_scripted.pt')
    print('[EXPORT] TorchScript saved')
except Exception as e:
    print('[EXPORT] TorchScript skipped:', e)
# Drive backup
if IN_COLAB and os.path.exists('/content/drive'):
    drive_out = Path('/content/drive/MyDrive/L3_VFA_PMA_runs')/export_dir.name
    drive_out.mkdir(parents=True, exist_ok=True)
    for f in export_dir.glob('*'):
        shutil.copy2(f, drive_out/f.name)
    print('[EXPORT] Backed up to Drive:', drive_out)


[EXPORT] Saved checkpoints to outputs/model_export_20251128-005510
[EXPORT] TorchScript saved


In [352]:
# 11) Determinizm & örnek batch kontrolü\nimport random, numpy as np\nseed = 42\nrandom.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\nif torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\nprint('[SEED] Set to', seed)\ntry:\n    if train_loader:\n        b = next(iter(train_loader))\n        print('[BATCH] image', b['image'].shape, 'psoas_left', b['psoas_left'].shape)\n    else:\n        print('[BATCH] Loader yok')\nexcept Exception as e:\n    print('[BATCH] Hata:', e)\n

In [353]:
# 12) Hızlı doğrulama metrikleri (Dice) – eğitim sonrası çalıştırın\nimport numpy as np\nfrom torch.nn.functional import one_hot\n\nif val_loader and len(val_ds)>0:\n    model.eval()\n    dices_left=[]; dices_right=[]\n    with torch.no_grad():\n        for batch in val_loader:\n            x = batch['image'].to(device)\n            y_l = batch['psoas_left'].to(device)\n            y_r = batch['psoas_right'].to(device)\n            logits = model(x)  # [B,2,H,W]\n            prob = torch.softmax(logits, dim=1)\n            pred_l = (prob[:,0]>0.5).float()\n            pred_r = (prob[:,1]>0.5).float()\n            # Dice = 2TP / (2TP + FP + FN) = 2*sum(pred*gt)/(sum(pred)+sum(gt)+1e-6)\n            def dice_bin(pred, gt):\n                inter = (pred*gt).sum()\n                return (2*inter)/(pred.sum()+gt.sum()+1e-6)\n            dices_left.append(dice_bin(pred_l, y_l))\n            dices_right.append(dice_bin(pred_r, y_r))\n    dl = float(torch.stack(dices_left).mean()) if dices_left else float('nan')\n    dr = float(torch.stack(dices_right).mean()) if dices_right else float('nan')\n    print(f'[VAL] Dice Left={dl:.4f} Right={dr:.4f}')\nelse:\n    print('[VAL] Val loader yok, metrik atlandı.')\n

## 11+ İleri Ayarlar & Tam Eğitim Rehberi
Tam eğitim öncesi öneriler:

Model Genişletme:
- Hafif: channels=(16,32,64) (mevcut)
- Orta: channels=(32,64,128), strides=(2,2)
- Büyük: channels=(32,64,128,256), strides=(2,2,2) (daha fazla VRAM)

CONFIG Ölçeklendirme:
```python
CONFIG['MAX_CASES'] = None      # Tüm AMOS22 vakaları
CONFIG['MAX_EPOCHS'] = 60       # Tam eğitim
CONFIG['BATCH_SIZE'] = 4        # VRAM’e göre 2/4/8
CONFIG['MERGE_STRATEGY'] = 'UNION'  # TS + C2C bütünleştirme (varsa)
```

Augmentasyon İyileştirme:
- Ekleyebilirsiniz: RandGaussianNoiseD, RandAdjustContrastD
- Eğer gürültü artıyorsa augmentasyonu azaltın.

Performans İpuçları:
- İlk TotalSegmentator çalışması ağırlık indirir (2GB). İkinci kez çok daha hızlı.
- GPU bellek hatası: BATCH_SIZE’i düşürün, channels listesini daraltın.
- Dice yavaş toparlıyorsa: lr=5e-4 veya AdamW -> SGD momentum=0.9 deneyebilirsiniz.

Val Metrik Hedefleri (mini test):
- Dice Left/Right > 0.80 (küçük veri ile)
- Tam eğitimde >0.90 beklenir.


In [354]:
# ============================================================
# AMOS22 + TotalSegmentator + Comp2Comp TEACHER ENTEGRASYONLU
# L3 VFA/PMA EĞİTİM PIPELINE (TEK BLOK)
# ============================================================
# Bu hücre:
#  - Drive yapılarını tekrar tanımlar (AMOS22, TS, C2C, WORK)
#  - AMOS CT NIfTI dosyalarını bulur
#  - TS öğretmeniyle L3 slice + maskeleri çıkarır
#  - VARSA Comp2Comp teacher maskelerini (VAT / SAT / psoas) OVERRIDE eder
#  - 4-kanallı (bg, VAT, SAT, psoas) 2D UNet için MONAI Dataset/DataLoader kurar
#  - 60 epoch training döngüsü (RUN_TRAINING ile kontrol) tanımlar
#  - Overlay görselleştirici ile teacher + model tahminini gösterir
# ============================================================

import os, sys, math, random, textwrap, subprocess
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from monai.networks.nets import UNet
from monai.transforms import (
    Compose, ScaleIntensityRange, EnsureChannelFirst, Resize,
    RandFlip, RandRotate, RandZoom
)

# Colab ortamında çalışıyorsak Drive modülü mevcut olacaktır; yerel çalışmada try/except ile koru
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

# ------------------------------
# 1) Drive ve klasör yapısı
# ------------------------------
if IN_COLAB and drive is not None:
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    # Yerel çalışma için fallback
    DRIVE_ROOT = Path.cwd()

AMOS_ROOT   = DRIVE_ROOT / 'AMOS22'                      # AMOS22 NIfTI kökü
if (AMOS_ROOT / 'imagesTr').exists():
    NIFTI_ROOT = AMOS_ROOT / 'imagesTr'
else:
    NIFTI_ROOT = AMOS_ROOT

TS_ROOT     = DRIVE_ROOT / 'TS_teachers_AMOS22'          # TotalSegmentator çıktı kökü
C2C_ROOT    = DRIVE_ROOT / 'C2C_teachers_AMOS22'         # Comp2Comp teacher .npy kökü
WORK_ROOT   = DRIVE_ROOT / 'AMOS22_training_work'
MODELS_DIR  = WORK_ROOT / 'models'
LOGS_DIR    = WORK_ROOT / 'logs'
OVERLAY_DIR = WORK_ROOT / 'overlays'

for p in [TS_ROOT, C2C_ROOT, WORK_ROOT, MODELS_DIR, LOGS_DIR, OVERLAY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('📁 NIFTI_ROOT :', NIFTI_ROOT)
print('📁 TS_ROOT    :', TS_ROOT)
print('📁 C2C_ROOT   :', C2C_ROOT)
print('📁 WORK_ROOT  :', WORK_ROOT)

# ------------------------------
# 2) AMOS CT NIfTI bul
# ------------------------------
def find_amos_ct_files(root: Path):
    nii_list = sorted(list(root.glob('*.nii.gz')))
    if not nii_list:
        nii_list = sorted(list(root.rglob('*.nii.gz')))
    return nii_list

ct_files = find_amos_ct_files(NIFTI_ROOT)
print(f'🔎 Bulunan AMOS CT sayısı: {len(ct_files)}')
for f in ct_files[:5]:
    print('  -', f)

if not ct_files:
    print(textwrap.dedent('''
    ⚠️ AMOS22 için NIfTI CT bulunamadı.
    Beklenen yapı örnekleri:
      /MyDrive/AMOS22/imagesTr/*.nii.gz
      veya
      /MyDrive/AMOS22/*.nii.gz
    '''))

# ------------------------------
# 3) Yardımcı fonksiyonlar
# ------------------------------
def load_nifti(path: Path):
    img = nib.load(str(path))
    data = img.get_fdata()
    return data, img.affine, img.header

def pick_l3_like_slice(ts_case_dir: Path):
    """
    TS vertebra maskelerinden L3'e yakın bir slice seçmek için basit heuristik:
      - vertebrae_lumbar varsa oradan, yoksa vertebrae_thoracic'ten
      - en geniş maskeye sahip slice = L3 proxy
    """
    cand_files = [
        ts_case_dir / 'vertebrae_lumbar.nii.gz',
        ts_case_dir / 'vertebrae_thoracic.nii.gz',
    ]
    for cf in cand_files:
        if cf.exists():
            vol, _, _ = load_nifti(cf)
            areas = vol.reshape(vol.shape[0], -1).sum(axis=1)
            if areas.max() > 0:
                z = int(areas.argmax())
                return z
    return None

TS_LABELS_WE_NEED = [
    'vertebrae_lumbar',
    'vertebrae_thoracic',
    'abdominal_wall',
    'muscle',
    'subcutaneous_fat',
    'torso_fat',
]

# ------------------------------
# 4) C2C override + TS teacher ile slice çıkarma
# ------------------------------
def extract_teacher_masks_for_case(ct_path: Path):
    """
    AMOS CT + TS + (VARSA) Comp2Comp teacher birleşimi ile
    L3 HU slice + VAT/SAT/psoas/fasya proxy maskelerini üretir.

    Öncelik sırası:
      - Fasya: TS abdominal_wall veya HU tabanlı body
      - VAT/SAT/psoas:
           * Eğer C2C_<CASE>_*.npy varsa → C2C override
           * Yoksa HU+TS ile approx teacher
    """
    case_id = ct_path.stem
    ts_case_dir = TS_ROOT / case_id
    if not ts_case_dir.exists():
        # Bu vakaya TS çalışmamış
        return None

    # CT HU hacmi
    hu, aff, hdr = load_nifti(ct_path)
    hu = hu.astype(np.float32)

    # L3 benzeri slice
    z = pick_l3_like_slice(ts_case_dir)
    if z is None or z < 0 or z >= hu.shape[0]:
        print(f'⚠️ {case_id}: L3 benzeri slice bulunamadı.')
        return None

    hu_slice = hu[z]

    # TS maskelerini oku
    ts_masks = {}
    for lbl in TS_LABELS_WE_NEED:
        f = ts_case_dir / f'{lbl}.nii.gz'
        if f.exists():
            vol, _, _ = load_nifti(f)
            ts_masks[lbl] = (vol[z] > 0).astype(np.uint8)
        else:
            ts_masks[lbl] = None

    # Fasya proxy (her durumda TS/body tabanlı kalsın)
    fascia_proxy = ts_masks.get('abdominal_wall', None)
    if fascia_proxy is None:
        body_mask = (hu_slice > -300).astype(np.uint8)
        fascia_proxy = body_mask

    # Kas maskesi
    muscle_mask = ts_masks.get('muscle', None)
    if muscle_mask is None:
        muscle_mask = ((hu_slice >= -29) & (hu_slice <= 150)).astype(np.uint8)

    # HU yağ bandı
    fat_band = ((hu_slice >= -190) & (hu_slice <= -30)).astype(np.uint8)

    # --- Varsayılan VAT/SAT/psoas (TS + HU approx) ---
    sat_mask_default = ts_masks.get('subcutaneous_fat', None)
    if sat_mask_default is None:
        sat_mask_default = (fat_band * (1 - fascia_proxy)).astype(np.uint8)

    vat_mask_default = (fat_band * fascia_proxy).astype(np.uint8)

    H, W = hu_slice.shape
    cx = W // 2
    y0, y1 = int(H * 0.55), int(H * 0.95)
    x_margin = int(W * 0.15)
    psoas_band = np.zeros_like(muscle_mask, dtype=np.uint8)
    psoas_band[y0:y1, :] = 1
    psoas_band[:, cx - x_margin:cx + x_margin] = 0
    psoas_proxy_default = (muscle_mask & psoas_band).astype(np.uint8)

    # ------------------------------------------------
    #  C2C OVERRIDE BLOĞU (VARSA)
    # ------------------------------------------------
    vat_mask   = vat_mask_default.copy()
    sat_mask   = sat_mask_default.copy()
    psoas_mask = psoas_proxy_default.copy()

    def load_c2c_mask(name: str):
        """
        C2C_ROOT altında şu isimleri arar:
          <case_id>_name.npy
        name ∈ {'vat','sat','psoas_left','psoas_right'}
        3D (Z,Y,X) gelirse aynı z slice'ı alır,
        2D (Y,X) gelirse direkt kullanır.
        """
        f = C2C_ROOT / f'{case_id}_{name}.npy'
        if not f.exists():
            return None
        arr = np.load(f)
        if arr.ndim == 3:
            if z >= arr.shape[0]:
                # slice aralığı uymuyorsa boş dön
                return None
            return (arr[z] > 0).astype(np.uint8)
        elif arr.ndim == 2:
            return (arr > 0).astype(np.uint8)
        else:
            return None

    c2c_vat = load_c2c_mask('vat')
    c2c_sat = load_c2c_mask('sat')
    c2c_pl  = load_c2c_mask('psoas_left')
    c2c_pr  = load_c2c_mask('psoas_right')

    # C2C varsa override et
    c2c_used = False

    if c2c_vat is not None:
        vat_mask = c2c_vat.copy()
        c2c_used = True
    if c2c_sat is not None:
        sat_mask = c2c_sat.copy()
        c2c_used = True
    if c2c_pl is not None or c2c_pr is not None:
        pl = c2c_pl if c2c_pl is not None else np.zeros_like(psoas_mask)
        pr = c2c_pr if c2c_pr is not None else np.zeros_like(psoas_mask)
        psoas_mask = ((pl > 0) | (pr > 0)).astype(np.uint8)
        c2c_used = True

    # Küçük bir log:
    if c2c_used:
        print(f'✅ {case_id}: C2C teacher override kullanıldı.')
    else:
        print(f'ℹ️ {case_id}: C2C bulunamadı, TS+HU approx kullanılıyor.')

    return {
        'case_id': case_id,
        'z': z,
        'hu_slice': hu_slice,
        'fascia_proxy': fascia_proxy,
        'muscle_mask': muscle_mask,
        'vat_mask': vat_mask,
        'sat_mask': sat_mask,
        'psoas_proxy': psoas_mask,
        'c2c_used': c2c_used,
    }

# ------------------------------
# 5) Tüm vakalar için teacher_slices oluştur
# ------------------------------
teacher_slices = []
for ct_path in tqdm(ct_files, desc='Teacher slice (TS + C2C) çıkarılıyor'):
    res = extract_teacher_masks_for_case(ct_path)
    if res is not None:
        teacher_slices.append(res)

print(f'✅ L3 benzeri teacher slice sayısı: {len(teacher_slices)}')
print(f'   (C2C override kullanılan vaka sayısı: {sum(int(x["c2c_used"]) for x in teacher_slices)})')

if len(teacher_slices) < 4:
    print("⚠️ 4'ten az slice var, eğitim çok sınırlı kalır.")

# ------------------------------
# 6) MONAI Dataset / DataLoader
# ------------------------------
IMG_SIZE = 256  # 512→256 downsample; hız ve RAM için

from monai.transforms import Resize as MonaiResize

class L3VFADataset(Dataset):
    """
    Girdi:  HU slice (512x512 civarı) → (1,IMG,IMG)
    Çıktı:  Label map (IMG,IMG) 0:bg, 1:VAT, 2:SAT, 3:psoas
    """
    def __init__(self, items, augment=False):
        self.items = items
        self.augment = augment
        self.tx_img = Compose([
            EnsureChannelFirst(),  # (H,W) → (1,H,W)
            ScaleIntensityRange(a_min=-1000.0, a_max=1000.0, b_min=0.0, b_max=1.0, clip=True),
            MonaiResize((IMG_SIZE, IMG_SIZE)),
        ])
        self.tx_lbl = MonaiResize((IMG_SIZE, IMG_SIZE), mode='nearest')
        self.aug = Compose([
            RandFlip(spatial_axis=1, prob=0.5),
            RandRotate(range_x=math.pi/36, prob=0.3),
            RandZoom(min_zoom=0.9, max_zoom=1.1, prob=0.3),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        itm = self.items[idx]
        hu = itm['hu_slice']
        vat = itm['vat_mask']
        sat = itm['sat_mask']
        psoas = itm['psoas_proxy']

        img = self.tx_img(hu.astype(np.float32))  # (1,IMG,IMG)

        # maskeleri de resize
        vat_t   = self.tx_lbl(torch.from_numpy(vat[None, None, ...].astype(np.float32)))
        sat_t   = self.tx_lbl(torch.from_numpy(sat[None, None, ...].astype(np.float32)))
        psoas_t = self.tx_lbl(torch.from_numpy(psoas[None, None, ...].astype(np.float32)))

        vat_r   = vat_t[0,0].round().numpy().astype(np.uint8)
        sat_r   = sat_t[0,0].round().numpy().astype(np.uint8)
        psoas_r = psoas_t[0,0].round().numpy().astype(np.uint8)

        # Label haritası
        label = np.zeros_like(vat_r, dtype=np.int64)
        label[vat_r > 0] = 1
        label[sat_r > 0] = 2
        label[psoas_r > 0] = 3

        img_t = torch.from_numpy(img.astype(np.float32))
        lbl_t = torch.from_numpy(label)

        if self.augment:
            stacked = torch.cat([img_t, lbl_t.unsqueeze(0).float()], dim=0)
            stacked = self.aug(stacked)
            img_t = stacked[0:1]
            lbl_t = stacked[1].round().long()

        return img_t, lbl_t

# Train / val split
random.shuffle(teacher_slices)
n_total = len(teacher_slices)
n_val = max(1, int(0.2 * n_total))
val_items = teacher_slices[:n_val]
train_items = teacher_slices[n_val:]

train_ds = L3VFADataset(train_items, augment=True)
val_ds   = L3VFADataset(val_items, augment=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(f'📊 Train slice sayısı: {len(train_ds)}')
print(f'📊 Val   slice sayısı: {len(val_ds)}')

# ------------------------------
# 7) UNet modeli + eğitim döngüsü
# ------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('⚙️ Device:', device)

model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=4,   # 0:bg, 1:VAT, 2:SAT, 3:psoas
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()
    running_loss = 0.0
    n_batches = 0
    with torch.set_grad_enabled(train):
        for img, lbl in loader:
            img = img.to(device)
            lbl = lbl.to(device)
            if train:
                optimizer.zero_grad()
            out = model(img)
            loss = criterion(out, lbl)
            if train:
                loss.backward()
                optimizer.step()
            running_loss += loss.item()
            n_batches += 1
    return running_loss / max(1, n_batches)

EPOCHS = 60
RUN_TRAINING = False  # 👉 Eğitimi başlatmak için True yap

best_val = float('inf')
if RUN_TRAINING:
    for epoch in range(1, EPOCHS + 1):
        tr_loss = run_epoch(train_loader, train=True)
        val_loss = run_epoch(val_loader, train=False)
        print(f'[{epoch:03d}/{EPOCHS}] train={tr_loss:.4f}  val={val_loss:.4f}')
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), MODELS_DIR / 'best_amos_ts_c2c_vfa.pth')
            print('  🔥 Yeni en iyi val loss, model kaydedildi.')
    torch.save(model.state_dict(), MODELS_DIR / 'last_amos_ts_c2c_vfa.pth')
    print('✅ Eğitim tamamlandı. En iyi val loss:', best_val)
else:
    print('ℹ️ RUN_TRAINING=False, şu an sadece altyapı kuruldu. Eğitime başlamak için RUN_TRAINING=True yapıp hücreyi tekrar çalıştır.')

# ------------------------------
# 8) Overlay görselleştirici
# ------------------------------
from monai.transforms import Resize as MonaiResize

def plot_overlay_example(idx=0, use_trained_model=False):
    if idx < 0 or idx >= len(teacher_slices):
        print('⚠️ Geçersiz idx.')
        return
    itm = teacher_slices[idx]
    hu = itm['hu_slice']
    vat = itm['vat_mask']
    sat = itm['sat_mask']
    psoas = itm['psoas_proxy']
    fascia = itm['fascia_proxy']

    pred_mask = None
    if use_trained_model and (MODELS_DIR / 'best_amos_ts_c2c_vfa.pth').exists():
        model.load_state_dict(torch.load(MODELS_DIR / 'best_amos_ts_c2c_vfa.pth', map_location=device))
        model.eval()
        with torch.no_grad():
            img = hu.astype(np.float32)
            img = np.clip((img + 1000.0) / 2000.0, 0.0, 1.0)
            img_t = torch.from_numpy(img[None, None, ...]).to(device)
            img_t = MonaiResize((IMG_SIZE, IMG_SIZE))(img_t)
            out = model(img_t)
            pred = out.argmax(dim=1)[0].cpu().numpy()
            # geri 512x512'e basit nearest upsample:
            t = torch.from_numpy(pred[None, None, ...].astype(np.float32))
            t = MonaiResize(hu.shape)(t)
            pred_mask = t[0,0].round().numpy().astype(np.int64)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(hu, cmap='gray')
    axes[0].set_title(f"CT HU (case: {itm['case_id']}, z={itm['z']})")
    axes[0].axis('off')

    overlay = np.stack([hu, hu, hu], axis=-1)
    # Fasya (proxy) kırmızı
    overlay[fascia > 0, 0] = overlay[fascia > 0, 0].max()
    # VAT yeşil
    overlay[vat > 0, 1] = overlay[vat > 0, 1].max()
    # SAT mavi
    overlay[sat > 0, 2] = overlay[sat > 0, 2].max()
    # Psoas sarı
    overlay[psoas > 0, 0] = overlay[psoas > 0, 0].max()
    overlay[psoas > 0, 1] = overlay[psoas > 0, 1].max()

    axes[1].imshow(overlay.astype(np.float32))
    axes[1].set_title(f"Teacher overlay (C2C override={'evet' if itm['c2c_used'] else 'hayır'})")
    axes[1].axis('off')

    if pred_mask is not None:
        pred_overlay = np.stack([hu, hu, hu], axis=-1)
        pred_overlay[pred_mask == 1, 1] = pred_overlay[pred_mask == 1, 1].max()  # VAT
        pred_overlay[pred_mask == 2, 2] = pred_overlay[pred_mask == 2, 2].max()  # SAT
        pred_overlay[pred_mask == 3, 0] = pred_overlay[pred_mask == 3, 0].max()
        pred_overlay[pred_mask == 3, 1] = pred_overlay[pred_mask == 3, 1].max()
        axes[2].imshow(pred_overlay.astype(np.float32))
        axes[2].set_title('Model tahmini overlay')
        axes[2].axis('off')
    else:
        axes[2].text(0.5, 0.5, 'Model tahmini yok\n(RUN_TRAINING=False veya model bulunamadı)',
                     ha='center', va='center', fontsize=10)
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()

print('\n➡️ Altyapı hazır.')
print(' - Eğitime başlamak için EPOCHS ve RUN_TRAINING=True ayarla, hücreyi yeniden çalıştır.')
print(' - Eğitim sonrası örnek görmek için: plot_overlay_example(idx=0, use_trained_model=True)')


📁 NIFTI_ROOT : /Users/alperenogras/Desktop/L3_SO_ANALYSIS/notebooks/AMOS22
📁 TS_ROOT    : /Users/alperenogras/Desktop/L3_SO_ANALYSIS/notebooks/TS_teachers_AMOS22
📁 C2C_ROOT   : /Users/alperenogras/Desktop/L3_SO_ANALYSIS/notebooks/C2C_teachers_AMOS22
📁 WORK_ROOT  : /Users/alperenogras/Desktop/L3_SO_ANALYSIS/notebooks/AMOS22_training_work
🔎 Bulunan AMOS CT sayısı: 0

⚠️ AMOS22 için NIfTI CT bulunamadı.
Beklenen yapı örnekleri:
  /MyDrive/AMOS22/imagesTr/*.nii.gz
  veya
  /MyDrive/AMOS22/*.nii.gz



Teacher slice (TS + C2C) çıkarılıyor: 0it [00:00, ?it/s]

✅ L3 benzeri teacher slice sayısı: 0
   (C2C override kullanılan vaka sayısı: 0)
⚠️ 4'ten az slice var, eğitim çok sınırlı kalır.


ValueError: num_samples should be a positive integer value, but got num_samples=0